# Verify & Benchmark Robot Segmentation (Steps 4–7)

Trains or loads single-frame predictors (`SmallSegUNet` and `YOLO26SemanticNet`) on offline teacher labels, visualizes moving and paused validation frames side by side with zoomed views and an embedded review video, and benchmarks paced 10 Hz latency and freshness.

In [ ]:
pack_path = "/Users/lboloni/Documents/Develop/Data/BerryPicker-Demopacks/random-both-cameras-video"
result_dir = "/Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/RobotSegmentation_pilot"
camera = "dev2"
teacher = "motion"
input_size = 256
epochs = 12
batch_size = 16
frames_per_episode = 80   # pose-balanced training frames per episode (None -> all retained)
train_episodes = None     # None -> all 6 train episodes in split.json
val_episodes = None       # None -> all 2 val episodes in split.json
yolo_weights = "yolo26n-seg.yaml"   # or "yolo26n-seg.pt" for pretrained init
retrain = False
benchmark_duration_s = 5.0
review_max_frames = 150
device = "auto"
seed = 0

In [ ]:
import json
import sys
from pathlib import Path

src_root = Path.cwd().resolve()
while src_root.name != "src" and src_root.parent != src_root:
    if (src_root / "_experimental" / "_2026_11_robot_segmentation").is_dir():
        break
    src_root = src_root.parent
if not (src_root / "_experimental" / "_2026_11_robot_segmentation").is_dir():
    src_root = Path("/Users/lboloni/Documents/Develop/Github/BerryPicker/BerryPicker/src")
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, Video, display

from _experimental._2026_11_robot_segmentation.baselines import MedianBackgroundModel, StaticMaskControl
from _experimental._2026_11_robot_segmentation.benchmark import run_paced_benchmark
from _experimental._2026_11_robot_segmentation.data import (
    camera_video_path, detect_pause_flags, frame_motion_scores,
    load_label_png, load_split, make_episode_split, read_video_frames,
    sample_pose_balanced_indices,
)
from _experimental._2026_11_robot_segmentation.evaluate import evaluate_predictor, pause_stability
from _experimental._2026_11_robot_segmentation.models import load_segmenter
from _experimental._2026_11_robot_segmentation.train import load_teacher_samples, train_semantic_student
from _experimental._2026_11_robot_segmentation.viz import (
    make_comparison_strip, make_zoom_crop, plot_latency_comparison,
    plot_training_curves, write_review_video,
)

out = Path(result_dir)
split = load_split(out / "split.json") if (out / "split.json").is_file() else make_episode_split(pack_path, camera=camera)
tr_eps = list(train_episodes) if train_episodes is not None else split.train
va_eps = list(val_episodes) if val_episodes is not None else split.val

def show_rgb(rgb):
    bgr = cv2.cvtColor(np.asarray(rgb, dtype=np.uint8), cv2.COLOR_RGB2BGR)
    display(Image(data=cv2.imencode(".png", bgr)[1].tobytes()))

bg = MedianBackgroundModel.load(out / "median_background" / camera)
static = StaticMaskControl.load(out / "static_control" / camera)
print("Train eps:", tr_eps)
print("Val eps  :", va_eps)

## Steps 4–5 — Train (or Load) Single-Frame Students on Matched Frames

In [ ]:
frame_filter = None
if frames_per_episode is not None:
    frame_filter = {}
    for ep in tr_eps:
        ids, frms = read_video_frames(camera_video_path(pack_path, ep, camera))
        chosen = sample_pose_balanced_indices(frms, count=frames_per_episode, pause_fraction=0.2, seed=seed)
        frame_filter[ep] = [ids[i] for i in chosen]

tr_imgs, tr_lbls, tr_recs = load_teacher_samples(
    pack_path, out / "teachers", teacher, camera, tr_eps, frame_filter=frame_filter
)
va_imgs, va_lbls, _ = load_teacher_samples(
    pack_path, out / "teachers", teacher, camera, va_eps
)
print(f"Loaded {len(tr_imgs)} training frames and {len(va_imgs)} validation frames (teacher={teacher})")

candidates = [
    ("SmallSegUNet", "unet", {"base_channels": 24}),
    ("YOLO26-Sem", "yolo26_semantic", {"weights_or_cfg": yolo_weights, "decoder_channels": 32}),
]
summaries = {}
student_dirs = {}
for label_name, arch, kwargs in candidates:
    run_dir = out / "models" / f"{arch}_{teacher}_{camera}"
    sum_path = run_dir / "training_summary.json"
    if retrain or not (run_dir / "best" / "config.json").is_file():
        summaries[label_name] = train_semantic_student(
            tr_imgs, tr_lbls, va_imgs, va_lbls,
            out_dir=run_dir, arch=arch, arch_kwargs=kwargs,
            camera_id=camera, input_size=input_size, epochs=epochs,
            batch_size=batch_size, seed=seed, device=device,
            split=split.to_dict(), train_records=tr_recs,
        )
    else:
        summaries[label_name] = json.loads(sum_path.read_text(encoding="utf-8"))
    student_dirs[label_name] = run_dir / "best"

fig = plot_training_curves(summaries)
fig.savefig(out / f"training_curves_{camera}.png", dpi=130)
plt.show()

## Side-by-Side Validation Frames (Moving & Paused) + Review Video

In [ ]:
predictors = {
    "Median-BG": bg,
    "Static": static,
}
for name, pdir in student_dirs.items():
    predictors[name] = load_segmenter(pdir, device=device)

val_ep = va_eps[0]
ids, frms = read_video_frames(camera_video_path(pack_path, val_ep, camera))
motion = frame_motion_scores(frms)
pauses, _ = detect_pause_flags(motion)

# Pick moving and paused validation frames
move_idx = np.flatnonzero(~pauses)
pause_idx = np.flatnonzero(pauses)
picks = []
if len(move_idx):
    picks.extend(move_idx[np.linspace(0, len(move_idx) - 1, min(3, len(move_idx))).astype(int)].tolist())
if len(pause_idx):
    picks.extend(pause_idx[np.linspace(0, len(pause_idx) - 1, min(2, len(pause_idx))).astype(int)].tolist())

lbl_dir = out / "teachers" / teacher / camera / val_ep / "training_labels"
for idx in picks:
    fidx = ids[idx]
    rgb = frms[idx]
    masks = {"Median-BG": bg.segment(rgb)}
    t_file = lbl_dir / f"frame_{fidx:06d}.png"
    if t_file.is_file():
        masks["Teacher"] = load_label_png(t_file)
    for name in student_dirs:
        masks[name] = predictors[name].segment(rgb)
    badge = f"f{fidx} {'PAUSE' if pauses[idx] else 'MOVE'}"
    show_rgb(make_comparison_strip(rgb, masks, badge=badge))

# Pause stability across the full validation episode
p_rows = ["| Method | Pause runs | Pause frames | Flicker rate | Adj IoU | Max area drop |", "|---|---:|---:|---:|---:|---:|"]
for name, pred in predictors.items():
    seq = [pred.segment(f) for f in frms]
    st = pause_stability(seq, pauses)
    fl = f"{st['mean_pause_flicker_rate']:.5f}" if st['mean_pause_flicker_rate'] is not None else "n/a"
    ai = f"{st['mean_pause_adj_iou']:.4f}" if st['mean_pause_adj_iou'] is not None else "n/a"
    dr = f"{100*st['max_pause_area_drop_frac']:.1f}%" if st['max_pause_area_drop_frac'] is not None else "n/a"
    p_rows.append(f"| {name} | {st['n_pause_runs']} | {st['pause_frames']} | {fl} | {ai} | {dr} |")
display(Markdown("### Validation Episode Pause Stability (`" + val_ep + "`)\n\n" + "\n".join(p_rows)))

In [ ]:
review_mp4 = out / f"review_{val_ep}_{camera}.mp4"
video_predictors = {"Median-BG": bg}
for name in student_dirs:
    video_predictors[name] = predictors[name]

write_review_video(
    frms,
    video_predictors,
    review_mp4,
    fps=10.0,
    pause_flags=pauses,
    max_frames=review_max_frames,
)
print("Saved review video:", review_mp4)
display(Video(str(review_mp4), embed=True))

## Paced 10 Hz Latency & Freshness Benchmark (100 ms Tick Budget)

In [ ]:
v_dev2 = camera_video_path(pack_path, val_ep, camera)
v_other = camera_video_path(pack_path, val_ep, "dev3" if camera == "dev2" else "dev2")

bench_reports = {}
for name in ["Median-BG", *student_dirs.keys()]:
    bench_reports[f"{name} (1 cam)"] = run_paced_benchmark(
        [predictors[name]], [v_dev2],
        duration_s=benchmark_duration_s, target_hz=10.0, warmup_ticks=5,
        out_path=out / f"bench_{name.lower().replace('-', '_')}_1cam.json",
    )

# Also measure 2-camera workload per 100 ms tick for each neural student
if v_other.is_file():
    for name in student_dirs:
        bench_reports[f"{name} (2 cams)"] = run_paced_benchmark(
            [predictors[name], predictors[name]], [v_dev2, v_other],
            duration_s=benchmark_duration_s, target_hz=10.0, warmup_ticks=5,
            out_path=out / f"bench_{name.lower().replace('-', '_')}_2cam.json",
        )

b_rows = [
    "| Configuration | Cold start (ms) | Median tick (ms) | p95 (ms) | Max (ms) | Max age (ms) | Misses | Drops | 10 Hz Pass |",
    "|---|---:|---:|---:|---:|---:|---:|---:|:---:|",
]
for label, r in bench_reports.items():
    b_rows.append(
        f"| {label} | {r['cold_start_ms']:.1f} | {r['tick_total_ms']['median']:.1f} | "
        f"{r['tick_total_ms']['p95']:.1f} | {r['tick_total_ms']['max']:.1f} | "
        f"{r['frame_age_ms']['max']:.1f} | {r['deadline_misses']} | {r['dropped_frames']} | "
        f"{'PASS' if r['observed_10hz_pass'] else 'FAIL'} |"
    )
display(Markdown("\n".join(b_rows)))

fig = plot_latency_comparison(bench_reports)
fig.savefig(out / f"latency_{camera}.png", dpi=130)
plt.show()

## Independent Reference Evaluation (Runs When Validation Masks Are Present)

In [ ]:
eval_manifest = out / "eval_pack" / "eval_manifest.json"
if eval_manifest.is_file():
    mdata = json.loads(eval_manifest.read_text(encoding="utf-8"))
    has_all = bool(mdata["val"]) and all((eval_manifest.parent / it["mask_rel"]).is_file() for it in mdata["val"])
    if has_all:
        e_rows = ["| Method | IoU | Boundary F1 | Base Recall | Gripper Recall | Shadow Leak | Gap Open Rate |", "|---|---:|---:|---:|---:|---:|---:|"]
        for name, pred in predictors.items():
            res = evaluate_predictor(pred, eval_manifest, "val", out / "eval_val" / name, input_size=input_size)
            ov = res["overall"]
            fmt = lambda v: f"{v:.4f}" if v is not None else "n/a"
            e_rows.append(f"| {name} | {fmt(ov['mean_iou'])} | {fmt(ov['mean_boundary_f1'])} | {fmt(ov['mean_base_recall'])} | {fmt(ov['mean_gripper_recall'])} | {fmt(ov['mean_shadow_leakage'])} | {fmt(ov['jaw_gap_open_rate'])} |")
        display(Markdown("\n".join(e_rows)))
    else:
        print("Independent validation reference masks not drawn yet under:", eval_manifest.parent / "val")
else:
    print("Run Explore_RobotSegmentation.ipynb first to export eval_pack.")